In [1]:
from uuid import uuid4
from dotenv import load_dotenv
from pathlib import Path
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import os

In [2]:
llm = None
vector_store = None

In [3]:
CHUNK_SIZE = 1000
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTORSTORE_DIR = "D:\\Learning\\dataScience_bootcamp\\gen-ai\\vectorstore"
COLLECTION_NAME = "rag_experiment"

In [4]:
GROQ_API_KEY="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"

In [5]:
def initialize_components():
    print('in initialize components method.....')
    global llm, vector_store
    if llm is None:
        llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.9, max_tokens=500, api_key=GROQ_API_KEY)
    print("groq setup is done.")
    if vector_store is None:
        ef = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL,
            model_kwargs={"trust_remote_code": True}
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=ef,
            persist_directory=str(VECTORSTORE_DIR)
        )


In [6]:
initialize_components()
vector_store.reset_collection()

in initialize components method.....
groq setup is done.


In [7]:
def generate_answer(query):
    if not vector_store:
        raise RuntimeError("Vector database is not initialized ")

    retriever = vector_store.as_retriever()

    chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vector_store.as_retriever())
    result = chain.invoke({"question": query}, return_only_outputs=True)
    sources = result.get("sources", "")

    return result['answer'], sources

In [8]:
answer, sources = generate_answer("Who is Sunil's Brother?")
print(f"Answer: {answer}")
print(f"Sources: {sources}")

D:\installedApps\python_3_10_11\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


Answer: I don't know.

Sources: 


In [9]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("D://Learning//dataScience_bootcamp//gen-ai//rag_text.txt")
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " "],
        chunk_size=CHUNK_SIZE
    )
docs = text_splitter.split_documents(data)
uuids = [str(uuid4()) for _ in range(len(docs))]
vector_store.add_documents(docs, ids=uuids)

['6d49bc0d-5af5-44ad-b2b4-395cec5749c1']

In [13]:
answer, sources = generate_answer("Who is Sunil's Sister?")
print(f"Answer: {answer}")
print(f"Sources: {sources}")

Answer: Ramya is Sunil's sister.

Sources: D://Learning//dataScience_bootcamp//gen-ai//rag_text.txt


In [14]:
answer, sources = generate_answer("Who is Anil's Father?")
print(f"Answer: {answer}")
print(f"Sources: {sources}")

Answer: Anil's father is Krishna.

Sources: D://Learning//dataScience_bootcamp//gen-ai//rag_text.txt
